# Access Surface Water Inventory and Monitoring (SWIM) water extent
This notebook provides a basic overview on how to query the [SWIM Water Extent - Sentinel-1/2 - Global, Daily](https://geoservice.dlr.de/web/datasets/swim) products via its STAC API. 

SWIM Water Extent is a global surface water product at 10 m pixel spacing based on Sentinel-1/2 data. The data collection contains binary layers indicating open surface water for each Sentinel-1/2 scene. Clouds and cloud shadows are removed using [ukis-csmask](https://github.com/dlr-eoc/ukis-csmask) and are represented as NoData.

In [ ]:
import dask.diagnostics
import folium
import folium.plugins as plugins
import geopandas as gpd

from odc.stac import load
from pystac_client import Client
from shapely.geometry import shape

## Query STAC API
The easiest way to work with the data is to use the [Geoservice STAC API](https://geoservice.dlr.de/eoc/ogc/stac). The SWIM water extent products are located in the [SWIM_WE collection](https://geoservice.dlr.de/eoc/ogc/stac/v1/collections/SWIM_WE). In the following we use [pystac-client](https://github.com/stac-utils/pystac-client) to query the items in this collections by bounding box (*bbox*) and acquisition dates (*dates_from_to*). Optionally, we can also filter the items by properties such as when they have been updated.

In [ ]:
# search stac api
stacapi_endpoint = "https://geoservice.dlr.de/eoc/ogc/stac/v1"
collections = ["SWIM_WE"]
bbox = [33.851, -20.63, 35.859, -18.88]
dates_from_to = ["2019-03-01", "2019-03-31"]
query_filter = "updated > '2025-02-20T00:00:00.000' AND updated < '2026-10-28T10:00:00.000'"

catalog = Client.open(url=stacapi_endpoint, ignore_conformance=True)

search = catalog.search(
    collections=collections,
    bbox=bbox,
    datetime=dates_from_to,
    filter=query_filter,
    method="GET",
    filter_lang="cql2-text",
    max_items=100,
)

items = [item for item in search.items()]

print(f"Search returned {len(items)} items")

## Convert items to Geopandas Dataframe
We can convert the pystac items to a Geopandas Dataframe for further vector analysis or simply to save them to file in a range of GIS file formats.

In [ ]:
items_dict = [item.to_dict() for item in items]
items_gdf = gpd.GeoDataFrame.from_features(items_dict)
items_gdf.head(3)

## Visualize item footprints on map
We can use the Geopandas Dataframe to visualize item footprints on an interactive Leaflet (Folium) map.

In [ ]:
m = folium.Map(tiles="OpenStreetMap")
layer_control = folium.LayerControl(position="topright", collapsed=True)
fullscreen = plugins.Fullscreen()
marker_cluster = plugins.MarkerCluster(name="Item markers").add_to(m)

for item in items:
    geom_pt = shape(item.geometry).representative_point()
    folium.Marker(
        [geom_pt.y, geom_pt.x],
        popup=f"<a href={item.links[4].target} target='_blank' rel='noopener noreferrer'>{item.id}</a><br>"
        f"src_platform: {item.properties['src_platform']}<br>"
        f"datetime: {item.datetime}",
    ).add_to(marker_cluster)

style = {"fillColor": "#00000000", "color": "#0000ff", "weight": 1}

footprints = folium.GeoJson(items_gdf.to_json(), name="Item footprints", style_function=lambda x: style, control=True)

footprints.add_to(m)
layer_control.add_to(m)
fullscreen.add_to(m)
m.fit_bounds(m.get_bounds())
m

## Convert items to Xarray DataArray
We can convert the pystac items to a Xarray DataArray for further raster analysis, such as timeseries analysis or mosaicing. Note that here we set a lower resolution for faster visualization (the native resolution of the products is 0.0001).

In [ ]:
# specify nodata value and target resolution
nodata = 255
resolution = 0.001

# load items to xarray
data_array = load(
    items,
    crs="EPSG:4326",
    resolution=resolution,
    lon=(bbox[0], bbox[2]),
    lat=(bbox[1], bbox[3]),
    skip_broken_datasets=True,
    chunks={"latitude": 2048, "longitude": 2048},
)["data"]

print(data_array)

## Maximum water extent mosaic
Here we compute the maximum value along time dimension to mosaic all water extent products.

In [ ]:
# get maximum value along time dimension to mosaic water masks
mosaic = data_array.max(dim="time").fillna(nodata).astype("uint8")

# compute and plot mosaic
with dask.diagnostics.ProgressBar():
    data = mosaic.compute()
    data.plot(size=10, cmap="Blues", add_colorbar=False)

## Flood frequency and duration
Here we do some more timeseries analysis to compute relative water frequency and flood duration maps.

In [ ]:
# compute relative flood frequency
count = data_array.sum("time")
valid_obs = data_array.count("time")
frequency = (100 * count / valid_obs).fillna(nodata).astype("uint8")

with dask.diagnostics.ProgressBar():
    data = frequency.compute()
    data.plot(size=10, cmap="Blues", vmin=0, vmax=100, cbar_kwargs={"label": "Relative water frequency (%)"})

In [ ]:
# compute total flood duration with forward fill assumption
duration = data_array.ffill(dim="time").fillna(False).resample(time="1D").ffill().sum(dim="time").astype("uint8")

with dask.diagnostics.ProgressBar():
    data = duration.compute()
    data.plot(size=10, cmap="Blues", cbar_kwargs={"label": "Flood duration (days)"})